In [ ]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd

# ---------- User-configurable paths ----------
path_data_intermediate = "/path/to/intermediate"  # <-- set this
os.makedirs(path_data_intermediate, exist_ok=True)

parquet_out = os.path.join(path_data_intermediate, "CompustatCSegments.parquet")
csv_out     = os.path.join(path_data_intermediate, "CompustatCSegments.csv")

In [ ]:
#1 Load Compustat Data

SQL = """
SELECT
    a.*
FROM compseg.wrds_seg_customer AS a
WHERE a.srcdate >= DATE '2000-01-01'  -- ensure data from year 2000+
;
"""


In [ ]:
#2 Compustat Data Extraction From WRDS

db = wrds.Connection()  # prompts for credentials if needed
df = db.raw_sql(SQL, date_cols=["srcdate"])


In [ ]:
#3 Data Cleaning

# ------------ Rename srcdate -> datadate ------------
if "srcdate" in df.columns:
    df = df.rename(columns={"srcdate": "datadate"})

# (Optional) sort for readability
sort_cols = [c for c in ["gvkey", "datadate"] if c in df.columns]
if sort_cols:
    df = df.sort_values(sort_cols, kind="mergesort")

# ------------ Save ------------
df.to_csv(csv_out , index=False)
# Optional Parquet (handy for faster reloads)
df.to_parquet(parquet_out, index=False)

print("Saved:")
print(" -", csv_out )
print(" -", parquet_out)
print(df.head())